### KPIs: avg fare/km

In [13]:
from clickhouse_driver import Client

client = Client(
    host="localhost",
    port=int(9000),
    user="click",
    password="click",
    database="pl",
)

with open("../sql/avg_fare_km.sql", "r") as f:
    query = f.read()

df = client.query_dataframe(query)
# df

# debug

In [14]:
# df.hist()

In [15]:
# avg fare per kkm
df['avg_fare_per_km'] = df['total_fare'] / df['total_distance']

# get fleet utilization rate
df_fleet = df.groupby(by=["fleet_id"]).mean("avg_fare_per_km").reset_index()

In [16]:
# df_fleet

In [17]:
import plotly.express as px

# sort by ur
df_plot = df_fleet.sort_values("avg_fare_per_km", ascending=False).copy()

# fleet_id to str to keep only existing fleets on x
df_plot["fleet_id"] = df_plot["fleet_id"].astype(str)

# add ur %
df_plot["avg_fare_per_km"] = (df_plot["avg_fare_per_km"]).round(2)

fig = px.bar(
    df_plot,
    x="fleet_id",
    y="avg_fare_per_km",
    color="fleet_id",
    text="avg_fare_per_km",
    title="AVG fare/km",
    labels={
        "fleet_id": "Fleet ID",
        "avg_fare_per_km": "AVG fare/km",
    },
)

# txt improvement
fig.update_traces(
    texttemplate="%{text}$",
    textposition="outside",
    hovertemplate="<b>Fleet %{x}</b><br>AVG fare/km: %{text}$",
)

fig.update_layout(
    title_font_size=24,
    yaxis=dict(range=[0, 5]),
    legend_title_text="Fleet",
    legend=dict(font=dict(size=12)),
    template="plotly_white",
)

fig.show()